In [ ]:
# Runs to analyse: $NESYSY_RUNS (set by run.py), else output/full, else output/subset.
# Figures and tables are written to <runs>/paper_results/.
import os
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
RUNS = Path(os.environ.get("NESYSY_RUNS") or next(
    (ROOT / "output" / m for m in ("full", "subset") if (ROOT / "output" / m).exists()),
    ROOT / "output" / "subset",
))
# the paper shows only points with all 25 runs; a subset has fewer runs per point
MIN_RUNS = int(os.environ.get("NESYSY_MIN_RUNS", 25 if "full" in RUNS.parts else 1))
OUT = RUNS / "paper_results"
OUT.mkdir(parents=True, exist_ok=True)
DBS = sorted(RUNS.glob("**/mlflow.db"))


def search_runs(experiment):
    """mlflow.search_runs of an experiment, over all databases below RUNS."""
    frames = []
    for db in DBS:
        mlflow.set_tracking_uri(f"sqlite:///{db}")
        exp = mlflow.get_experiment_by_name(experiment)
        if exp is not None:
            frames.append(mlflow.search_runs(experiment_ids=[exp.experiment_id]))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


print(f"runs: {RUNS} ({len(DBS)} database(s)), at least {MIN_RUNS} run(s) per point")


In [ ]:
# Table 1: fixing vs filtering (runs of table1_fixing_vs_filtering.py, in tab1_runs.csv)
files = sorted(RUNS.glob("**/tab1_runs.csv"))
if not files:
    print("No Table 1 runs found, skipping.")
else:
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df["label"] = "h" + df["hidden_dim"].astype(str) + "_l" + df["num_hidden"].astype(str)

    metrics = ["accuracy", "constraint_violation_ratio", "hamming_distance", "n_unique_outputs", "n_total"]

    def summarize(group):
        out = {}
        for m in metrics:
            vals = group[m]
            out[f"{m}_mean"] = vals.mean()
            out[f"{m}_se"] = vals.std(ddof=1) / np.sqrt(len(vals))
        out["n_runs"] = len(group)
        out["n_deficit"] = group["deficit"].sum()
        return pd.Series(out)

    summary = df.groupby(["generator", "label"], sort=False).apply(summarize).reset_index()
    summary.to_csv(OUT / "table1.csv", index=False)
    display(summary)
